# Demographic Intelligence System
## Complete Training Pipeline with Bias Analysis

This notebook implements a production-grade system for age, gender, and emotion recognition with fairness analysis.

In [ ]:
# Import required libraries
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, applications
import cv2
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report, mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU Available: {tf.config.list_physical_devices('GPU')}")

## 1. Data Loading and EDA

In [ ]:
class DataLoader:
    """Handles UTKFace dataset loading and preprocessing"""
    
    def __init__(self, data_path='data/UTKFace/'):
        self.data_path = data_path
        self.images = []
        self.ages = []
        self.genders = []
        self.emotions = []
        
    def load_utkface(self, sample_size=None):
        """Load UTKFace dataset with age, gender, and emotion labels"""
        print("Loading UTKFace dataset...")
        
        # Emotion mapping (simplified - in production, use real emotion labels)
        emotion_map = {0: 'neutral', 1: 'happy', 2: 'sad', 3: 'angry', 
                      4: 'surprise', 5: 'fear', 6: 'disgust'}
        
        image_files = [f for f in os.listdir(self.data_path) if f.endswith('.jpg')]
        
        if sample_size:
            image_files = image_files[:sample_size]
        
        for idx, filename in enumerate(image_files):
            try:
                # UTKFace format: [age]_[gender]_[race]_[date].jpg
                parts = filename.split('_')
                age = int(parts[0])
                gender = int(parts[1])  # 0: male, 1: female
                
                # Load and preprocess image
                img_path = os.path.join(self.data_path, filename)
                img = cv2.imread(img_path)
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                img = cv2.resize(img, (224, 224))
                
                # Simulate emotion based on age and gender (for demonstration)
                # In production, use a dataset with real emotion labels
                if age < 18:
                    emotion = 1  # happy for young people
                elif age > 60:
                    emotion = 2  # sad for elderly
                else:
                    emotion = 0  # neutral for adults
                
                self.images.append(img)
                self.ages.append(age)
                self.genders.append(gender)
                self.emotions.append(emotion)
                
                if idx % 1000 == 0:
                    print(f"Loaded {idx}/{len(image_files)} images")
                    
            except Exception as e:
                continue
        
        return np.array(self.images), np.array(self.ages), np.array(self.genders), np.array(self.emotions)
    
    def create_dataframe(self):
        """Create pandas DataFrame for analysis"""
        df = pd.DataFrame({
            'age': self.ages,
            'gender': ['Male' if g == 0 else 'Female' for g in self.genders],
            'emotion': self.emotions
        })
        return df

In [ ]:
# Load dataset
data_loader = DataLoader(data_path='data/UTKFace/')
images, ages, genders, emotions = data_loader.load_utkface(sample_size=5000)

# Normalize images
images = images.astype('float32') / 255.0

print(f"\nDataset loaded successfully!")
print(f"Total samples: {len(images)}")
print(f"Image shape: {images[0].shape}")
print(f"Age range: {ages.min()} - {ages.max()}")
print(f"Gender distribution: Male={(genders==0).sum()}, Female={(genders==1).sum()}")

In [ ]:
# Exploratory Data Analysis
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Age distribution
axes[0, 0].hist(ages, bins=50, edgecolor='black', alpha=0.7)
axes[0, 0].set_title('Age Distribution', fontsize=14, fontweight='bold')
axes[0, 0].set_xlabel('Age')
axes[0, 0].set_ylabel('Frequency')

# Gender distribution
gender_counts = ['Male', 'Female']
gender_values = [(genders==0).sum(), (genders==1).sum()]
axes[0, 1].bar(gender_counts, gender_values, color=['blue', 'pink'], alpha=0.7)
axes[0, 1].set_title('Gender Distribution', fontsize=14, fontweight='bold')
axes[0, 1].set_ylabel('Count')

# Age by gender
male_ages = ages[genders==0]
female_ages = ages[genders==1]
axes[1, 0].boxplot([male_ages, female_ages], labels=['Male', 'Female'])
axes[1, 0].set_title('Age Distribution by Gender', fontsize=14, fontweight='bold')
axes[1, 0].set_ylabel('Age')

# Sample images
sample_indices = np.random.choice(len(images), 9, replace=False)
for i, idx in enumerate(sample_indices):
    ax = axes[1, 1].inset_axes([(i%3)*0.33, (2-i//3)*0.33, 0.33, 0.33])
    ax.imshow(images[idx])
    ax.set_title(f'Age: {ages[idx]}, Gender: {"M" if genders[idx]==0 else "F"}', fontsize=8)
    ax.axis('off')
axes[1, 1].axis('off')

plt.tight_layout()
plt.savefig('results/plots/eda_analysis.png', dpi=100, bbox_inches='tight')
plt.show()

## 2. Data Preprocessing and Augmentation

In [ ]:
def create_data_pipeline(images, ages, genders, emotions, batch_size=32):
    """Create TensorFlow data pipeline with augmentation"""
    
    # Split data
    X_train, X_temp, y_age_train, y_age_temp, y_gender_train, y_gender_temp, y_emo_train, y_emo_temp = train_test_split(
        images, ages, genders, emotions, test_size=0.3, random_state=42
    )
    
    X_val, X_test, y_age_val, y_age_test, y_gender_val, y_gender_test, y_emo_val, y_emo_test = train_test_split(
        X_temp, y_age_temp, y_gender_temp, y_emo_temp, test_size=0.5, random_state=42
    )
    
    # Data augmentation for training
    data_augmentation = keras.Sequential([
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.1),
        layers.RandomZoom(0.1),
        layers.RandomContrast(0.1),
    ])
    
    def augment_image(img, age, gender, emotion):
        img = data_augmentation(img, training=True)
        return img, {'age_output': age, 'gender_output': gender, 'emotion_output': emotion}
    
    # Create datasets
    train_dataset = tf.data.Dataset.from_tensor_slices((X_train, y_age_train, y_gender_train, y_emo_train))
    train_dataset = train_dataset.map(lambda x, a, g, e: augment_image(x, a, g, e))
    train_dataset = train_dataset.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    
    val_dataset = tf.data.Dataset.from_tensor_slices((X_val, y_age_val, y_gender_val, y_emo_val))
    val_dataset = val_dataset.map(lambda x, a, g, e: (x, {'age_output': a, 'gender_output': g, 'emotion_output': e}))
    val_dataset = val_dataset.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    
    test_dataset = tf.data.Dataset.from_tensor_slices((X_test, y_age_test, y_gender_test, y_emo_test))
    test_dataset = test_dataset.map(lambda x, a, g, e: (x, {'age_output': a, 'gender_output': g, 'emotion_output': e}))
    test_dataset = test_dataset.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    
    print(f"Training samples: {len(X_train)}")
    print(f"Validation samples: {len(X_val)}")
    print(f"Test samples: {len(X_test)}")
    
    return train_dataset, val_dataset, test_dataset, (X_test, y_age_test, y_gender_test, y_emo_test)

# Create data pipeline
train_dataset, val_dataset, test_dataset, test_data = create_data_pipeline(images, ages, genders, emotions)

## 3. Multi-Task Model Architecture

In [ ]:
def build_multi_task_model(input_shape=(224, 224, 3)):
    """Build multi-task learning model for age, gender, and emotion prediction"""
    
    # Base model (transfer learning)
    base_model = applications.ResNet50(
        weights='imagenet',
        include_top=False,
        input_shape=input_shape
    )
    base_model.trainable = False  # Freeze base model initially
    
    # Common feature extractor
    inputs = layers.Input(shape=input_shape)
    x = base_model(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(512, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)
    
    # Age prediction head (regression)
    age_output = layers.Dense(256, activation='relu')(x)
    age_output = layers.Dropout(0.2)(age_output)
    age_output = layers.Dense(1, name='age_output')(age_output)
    
    # Gender classification head (binary)
    gender_output = layers.Dense(128, activation='relu')(x)
    gender_output = layers.Dropout(0.2)(gender_output)
    gender_output = layers.Dense(1, activation='sigmoid', name='gender_output')(gender_output)
    
    # Emotion classification head (multi-class)
    emotion_output = layers.Dense(128, activation='relu')(x)
    emotion_output = layers.Dropout(0.2)(emotion_output)
    emotion_output = layers.Dense(7, activation='softmax', name='emotion_output')(emotion_output)
    
    # Create model
    model = models.Model(inputs=inputs, outputs=[age_output, gender_output, emotion_output])
    
    return model, base_model

# Build model
model, base_model = build_multi_task_model()
model.summary()

In [ ]:
# Compile model with custom losses
losses = {
    'age_output': 'mae',  # Mean Absolute Error for regression
    'gender_output': 'binary_crossentropy',
    'emotion_output': 'sparse_categorical_crossentropy'
}

loss_weights = {
    'age_output': 1.0,
    'gender_output': 0.5,
    'emotion_output': 0.8
}

metrics = {
    'age_output': ['mae'],
    'gender_output': ['accuracy', tf.keras.metrics.Precision(), tf.keras.metrics.Recall()],
    'emotion_output': ['accuracy']
}

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss=losses,
    loss_weights=loss_weights,
    metrics=metrics
)

print("Model compiled successfully!")

## 4. Training with Callbacks

In [ ]:
# Callbacks for training
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=10,
        restore_best_weights=True,
        verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=1e-6,
        verbose=1
    ),
    tf.keras.callbacks.ModelCheckpoint(
        'models/best_model.h5',
        monitor='val_loss',
        save_best_only=True,
        verbose=1
    ),
    tf.keras.callbacks.TensorBoard(
        log_dir='./logs',
        histogram_freq=1,
        update_freq='epoch'
    )
]

# Train the model
history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=50,
    callbacks=callbacks,
    verbose=1
)

## 5. Fine-tuning

In [ ]:
# Fine-tune the base model
base_model.trainable = True

# Freeze early layers
for layer in base_model.layers[:100]:
    layer.trainable = False

# Recompile with lower learning rate
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss=losses,
    loss_weights=loss_weights,
    metrics=metrics
)

# Continue training
history_finetune = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=30,
    callbacks=callbacks,
    verbose=1
)

## 6. Training Visualization

In [ ]:
# Plot training history
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# Combine histories
all_history = {}
for key in history.history.keys():
    if 'val' in key:
        all_history[key] = history.history[key] + history_finetune.history[key]
    else:
        all_history[key] = history.history[key] + history_finetune.history[key]

# Age loss
axes[0, 0].plot(all_history['age_output_loss'], label='Train', linewidth=2)
axes[0, 0].plot(all_history['val_age_output_loss'], label='Validation', linewidth=2)
axes[0, 0].set_title('Age Prediction Loss (MAE)', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Gender accuracy
axes[0, 1].plot(all_history['gender_output_accuracy'], label='Train', linewidth=2)
axes[0, 1].plot(all_history['val_gender_output_accuracy'], label='Validation', linewidth=2)
axes[0, 1].set_title('Gender Classification Accuracy', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Accuracy')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Emotion accuracy
axes[0, 2].plot(all_history['emotion_output_accuracy'], label='Train', linewidth=2)
axes[0, 2].plot(all_history['val_emotion_output_accuracy'], label='Validation', linewidth=2)
axes[0, 2].set_title('Emotion Recognition Accuracy', fontsize=12, fontweight='bold')
axes[0, 2].set_xlabel('Epoch')
axes[0, 2].set_ylabel('Accuracy')
axes[0, 2].legend()
axes[0, 2].grid(True, alpha=0.3)

# Combined loss
axes[1, 0].plot(all_history['loss'], label='Train Total Loss', linewidth=2)
axes[1, 0].plot(all_history['val_loss'], label='Validation Total Loss', linewidth=2)
axes[1, 0].set_title('Combined Loss', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('Loss')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Gender precision/recall
axes[1, 1].plot(all_history['gender_output_precision'], label='Precision', linewidth=2)
axes[1, 1].plot(all_history['gender_output_recall'], label='Recall', linewidth=2)
axes[1, 1].set_title('Gender Metrics', fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('Score')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

# Learning rate
axes[1, 2].plot(all_history['lr'] if 'lr' in all_history else [1e-3] * len(all_history['loss']), linewidth=2)
axes[1, 2].set_title('Learning Rate Schedule', fontsize=12, fontweight='bold')
axes[1, 2].set_xlabel('Epoch')
axes[1, 2].set_ylabel('Learning Rate')
axes[1, 2].set_yscale('log')
axes[1, 2].grid(True, alpha=0.3)

plt.suptitle('Model Training History', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('results/plots/training_history.png', dpi=100, bbox_inches='tight')
plt.show()

## 7. Model Evaluation

In [ ]:
# Evaluate on test set
test_results = model.evaluate(test_dataset, verbose=0)

print("="*50)
print("TEST SET EVALUATION RESULTS")
print("="*50)

# Extract predictions
X_test, y_age_test, y_gender_test, y_emo_test = test_data
predictions = model.predict(X_test)
age_preds = predictions[0].flatten()
gender_preds = (predictions[1].flatten() > 0.5).astype(int)
emotion_preds = np.argmax(predictions[2], axis=1)

# Age metrics
mae = mean_absolute_error(y_age_test, age_preds)
print(f"\n📊 AGE PREDICTION:")
print(f"   Mean Absolute Error: {mae:.2f} years")
print(f"   RMSE: {np.sqrt(np.mean((y_age_test - age_preds)**2)):.2f} years")
print(f"   R² Score: {1 - np.sum((y_age_test - age_preds)**2)/np.sum((y_age_test - np.mean(y_age_test))**2):.3f}")

# Gender metrics
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
gender_acc = accuracy_score(y_gender_test, gender_preds)
gender_prec = precision_score(y_gender_test, gender_preds)
gender_rec = recall_score(y_gender_test, gender_preds)
gender_f1 = f1_score(y_gender_test, gender_preds)

print(f"\n👥 GENDER CLASSIFICATION:")
print(f"   Accuracy: {gender_acc:.3f}")
print(f"   Precision: {gender_prec:.3f}")
print(f"   Recall: {gender_rec:.3f}")
print(f"   F1-Score: {gender_f1:.3f}")

# Emotion metrics
emo_acc = accuracy_score(y_emo_test, emotion_preds)
print(f"\n😊 EMOTION RECOGNITION:")
print(f"   Accuracy: {emo_acc:.3f}")
print(f"\n   Classification Report:")
emotion_labels = ['Neutral', 'Happy', 'Sad', 'Angry', 'Surprise', 'Fear', 'Disgust']
print(classification_report(y_emo_test, emotion_preds, target_names=emotion_labels))

In [ ]:
# Confusion matrices
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Gender confusion matrix
gender_cm = confusion_matrix(y_gender_test, gender_preds)
sns.heatmap(gender_cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Male', 'Female'], yticklabels=['Male', 'Female'])
axes[0].set_title('Gender Classification Confusion Matrix', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')

# Emotion confusion matrix
emo_cm = confusion_matrix(y_emo_test, emotion_preds)
sns.heatmap(emo_cm, annot=True, fmt='d', cmap='YlOrRd', ax=axes[1],
            xticklabels=emotion_labels, yticklabels=emotion_labels)
axes[1].set_title('Emotion Recognition Confusion Matrix', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Actual')
plt.xticks(rotation=45)
plt.yticks(rotation=0)

plt.tight_layout()
plt.savefig('results/plots/confusion_matrices.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# Age prediction visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter plot
axes[0].scatter(y_age_test, age_preds, alpha=0.5, s=10)
axes[0].plot([y_age_test.min(), y_age_test.max()], [y_age_test.min(), y_age_test.max()], 
             'r--', lw=2, label='Perfect Prediction')
axes[0].set_xlabel('Actual Age', fontsize=12)
axes[0].set_ylabel('Predicted Age', fontsize=12)
axes[0].set_title(f'Age Predictions vs Actual (MAE: {mae:.2f} years)', fontsize=14, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Error distribution
errors = age_preds - y_age_test
axes[1].hist(errors, bins=50, edgecolor='black', alpha=0.7)
axes[1].axvline(x=0, color='r', linestyle='--', linewidth=2)
axes[1].set_xlabel('Prediction Error (years)', fontsize=12)
axes[1].set_ylabel('Frequency', fontsize=12)
axes[1].set_title(f'Age Prediction Error Distribution (std: {errors.std():.2f})', fontsize=14, fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('results/plots/age_predictions.png', dpi=100, bbox_inches='tight')
plt.show()

## 8. Bias and Fairness Analysis

In [ ]:
def analyze_bias(ages, genders, age_preds, gender_preds, y_age_test, y_gender_test):
    """Comprehensive bias analysis across demographic groups"""
    
    bias_report = {}
    
    # Age group analysis
    age_groups = [(0, 18, 'Children'), (18, 30, 'Young Adult'), 
                  (30, 50, 'Adult'), (50, 65, 'Middle Age'), (65, 100, 'Senior')]
    
    print("="*60)
    print("BIAS AND FAIRNESS ANALYSIS REPORT")
    print("="*60)
    
    print("\n📊 PERFORMANCE BY AGE GROUP:")
    print("-"*40)
    
    age_metrics = []
    for min_age, max_age, group_name in age_groups:
        mask = (y_age_test >= min_age) & (y_age_test < max_age)
        if mask.sum() > 0:
            group_mae = mean_absolute_error(y_age_test[mask], age_preds[mask])
            group_gender_acc = accuracy_score(y_gender_test[mask], gender_preds[mask])
            
            print(f"  {group_name:15s} (n={mask.sum():4d}): Age MAE={group_mae:.2f}, Gender Acc={group_gender_acc:.3f}")
            
            age_metrics.append({
                'group': group_name,
                'count': mask.sum(),
                'age_mae': group_mae,
                'gender_acc': group_gender_acc
            })
    
    print("\n📊 PERFORMANCE BY GENDER:")
    print("-"*40)
    
    # Male performance
    male_mask = y_gender_test == 0
    male_age_mae = mean_absolute_error(y_age_test[male_mask], age_preds[male_mask])
    male_gender_acc = accuracy_score(y_gender_test[male_mask], gender_preds[male_mask])
    
    # Female performance
    female_mask = y_gender_test == 1
    female_age_mae = mean_absolute_error(y_age_test[female_mask], age_preds[female_mask])
    female_gender_acc = accuracy_score(y_gender_test[female_mask], gender_preds[female_mask])
    
    print(f"  Male   (n={male_mask.sum():4d}): Age MAE={male_age_mae:.2f}, Gender Acc={male_gender_acc:.3f}")
    print(f"  Female (n={female_mask.sum():4d}): Age MAE={female_age_mae:.2f}, Gender Acc={female_gender_acc:.3f}")
    
    # Bias metrics
    print("\n⚖️ BIAS METRICS:")
    print("-"*40)
    
    age_bias = abs(male_age_mae - female_age_mae)
    gender_bias = abs(male_gender_acc - female_gender_acc)
    
    print(f"  Age Prediction Bias (MAE difference): {age_bias:.3f} years")
    print(f"  Gender Classification Bias (Accuracy difference): {gender_bias:.3f}")
    
    # Disparate impact
    male_fp = np.sum((gender_preds[male_mask] == 1) & (y_gender_test[male_mask] == 0))
    female_fp = np.sum((gender_preds[female_mask] == 0) & (y_gender_test[female_mask] == 1))
    
    if female_fp > 0:
        disparate_impact = male_fp / female_fp
        print(f"  Disparate Impact Ratio: {disparate_impact:.3f}")
        if disparate_impact < 0.8:
            print("    ⚠️ Potential bias against females detected!")
        elif disparate_impact > 1.25:
            print("    ⚠️ Potential bias against males detected!")
        else:
            print("    ✓ Fairness criteria satisfied")
    
    return age_metrics

# Run bias analysis
bias_results = analyze_bias(ages, genders, age_preds, gender_preds, y_age_test, y_gender_test)

In [ ]:
# Visualize bias
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Age MAE by age group
groups = [m['group'] for m in bias_results]
mae_values = [m['age_mae'] for m in bias_results]
counts = [m['count'] for m in bias_results]

bars = axes[0].bar(groups, mae_values, color='steelblue', alpha=0.7)
axes[0].set_xlabel('Age Group', fontsize=12)
axes[0].set_ylabel('Mean Absolute Error (years)', fontsize=12)
axes[0].set_title('Age Prediction Performance by Age Group', fontsize=14, fontweight='bold')
axes[0].grid(True, alpha=0.3, axis='y')

# Add count annotations
for bar, count in zip(bars, counts):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1, 
                f'n={count}', ha='center', va='bottom', fontsize=10)

# Gender performance comparison
gender_metrics = {
    'Age MAE': [male_age_mae, female_age_mae],
    'Gender Acc': [male_gender_acc, female_gender_acc]
}

x = np.arange(len(gender_metrics))
width = 0.35

axes[1].bar(x - width/2, [gender_metrics['Age MAE'][0], gender_metrics['Gender Acc'][0]], 
           width, label='Male', color='blue', alpha=0.7)
axes[1].bar(x + width/2, [gender_metrics['Age MAE'][1], gender_metrics['Gender Acc'][1]], 
           width, label='Female', color='pink', alpha=0.7)

axes[1].set_xlabel('Metrics', fontsize=12)
axes[1].set_ylabel('Score', fontsize=12)
axes[1].set_title('Performance Comparison by Gender', fontsize=14, fontweight='bold')
axes[1].set_xticks(x)
axes[1].set_xticklabels(['Age MAE (years)', 'Gender Accuracy'])
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('results/plots/bias_analysis.png', dpi=100, bbox_inches='tight')
plt.show()

## 9. Save Models

In [ ]:
# Save the complete model
model.save('models/demographic_intelligence_model.h5')
print("✅ Full model saved to 'models/demographic_intelligence_model.h5'")

# Convert to TensorFlow Lite for edge deployment
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

with open('models/model.tflite', 'wb') as f:
    f.write(tflite_model)
print("✅ TensorFlow Lite model saved to 'models/model.tflite'")

# Save metrics
import json
metrics_summary = {
    'age_mae': float(mae),
    'gender_accuracy': float(gender_acc),
    'emotion_accuracy': float(emo_acc),
    'gender_precision': float(gender_prec),
    'gender_recall': float(gender_rec),
    'gender_f1': float(gender_f1),
    'bias_metrics': {
        'age_bias': float(age_bias),
        'gender_bias': float(gender_bias)
    }
}

with open('results/metrics/model_metrics.json', 'w') as f:
    json.dump(metrics_summary, f, indent=4)
    
print("✅ Metrics saved to 'results/metrics/model_metrics.json'")

print("\n🎉 Training pipeline completed successfully!")
print("="*50)